# Stokes equation - stabilized with Brezzi-Pitkäranta

In this tutorial we present how to solve the Stokes equation with [PyGeoN](https://github.com/compgeo-mox/pygeon) using an **equal-order** velocity-pressure pair: linear vector Lagrange elements for the velocity ${u}$ and linear (continuous) Lagrange elements for the pressure $p$. Since this pair does not satisfy the inf-sup (LBB) condition, we add a [Brezzi-Pitkäranta](https://link.springer.com/chapter/10.1007/978-3-663-14169-3_2) pressure stabilization.

Let $\Omega=(0,1)^2$ be a domain with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Defining the stress as $\sigma(u, p) = 2 \mu \epsilon(u) - p I$, we want to solve the following problem: find $({u}, p)$ such that
$$
\begin{align*}
&-\nabla \cdot \sigma(u, p)= {0} \\
&\nabla \cdot {u} = 0
\end{align*}
\quad \text{in } \Omega
$$
with boundary conditions
$$
{u} = {0} \quad \text{on } \partial_{top} \Omega \cup \partial_{bottom} \Omega,
$$
$$
 {\nu}\cdot \sigma(u, p) = {t} \quad \text{on } \partial_{right} \Omega, \qquad {t} = (1, 0)^\top,
$$
and a natural condition $ {\nu} \cdot \sigma(u, p)= {0}$ on $\partial_{left} \Omega$. So the flow is a pressure-driven channel flow: no-slip walls at the top and bottom, a prescribed normal stress pushing the fluid in on the right, and a free outflow on the left.

Since the velocity-pressure pair is equal-order (both linear), the discrete problem is stabilized by adding, for each pressure test/trial function, the term
$$
\delta \sum_{K} h_K^2 \int_K \nabla p \cdot \nabla q \, dx,
$$
with $\delta$ a stabilization parameter and $h_K$ the diameter of cell $K$.

We present *step-by-step* how to create the grid, declare the problem data, and finally solve the problem.

First we import some of the standard modules. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [1]:
import os

import numpy as np
import porepy as pp
import scipy.sparse as sps

import pygeon as pg

/home/elle/Dropbox/Work/Codes/porepy/src/porepy/utils/ui_and_logging.py:49: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm as progressbar_class  # type: ignore


We create now the grid. Since both the velocity and pressure use continuous Lagrange elements, here we consider a 2-dimensional triangular grid.

In [2]:
mesh_size = 0.1
dim = 2

sd = pg.unit_grid(dim, mesh_size, as_mdg=False)
sd.compute_geometry()

We define now the finite element spaces for all the variables. The velocity ${u}$ lives in the vector Lagrange space `VecLagrange1`, and the pressure $p$ in the (continuous) `Lagrange1` space. The piecewise constant `PwConstants` and piecewise linear (discontinuous) `PwLinears` spaces are auxiliary: they are only used to assemble the coupling term between velocity and pressure, since the divergence of a `VecLagrange1` function is a piecewise constant field.

In [3]:
key = "stokes"

# finite element spaces
vec_l1 = pg.VecLagrange1(key)  # velocity
l1 = pg.Lagrange1(key)  # pressure
p0 = pg.PwConstants()
p1 = pg.PwLinears()

# build the degrees of freedom
dofs = np.array([vec_l1.ndof(sd), l1.ndof(sd)])

With the following code we set the boundary conditions. We identify the top and bottom sides of $\partial \Omega$, where we impose the no-slip condition ${u} = {0}$ on both velocity components, and the right side, where we impose the normal stress ${t} = (1, 0)^\top$ as a natural boundary condition. No condition is imposed on the left side, which is therefore a natural, do-nothing boundary.

In [4]:
# identify the portions of the boundary to impose the boundary conditions
top = np.isclose(sd.nodes[1, :], 1)
bottom = np.isclose(sd.nodes[1, :], 0)
right = np.isclose(sd.face_centers[0, :], 1)

# no-slip: both velocity components are set to zero on the top and bottom
v_ess_dofs = np.hstack([np.logical_or(top, bottom)] * dim)
v_ess = np.zeros(vec_l1.ndof(sd))

ess_dofs = np.zeros(dofs.sum(), dtype=bool)
ess_dofs[: dofs[0]] = v_ess_dofs

ess_val = np.zeros(dofs.sum())
ess_val[: dofs[0]] = v_ess

# prescribed normal stress t = (1, 0) on the right side, natural condition
# on the left side
rhs_v = -vec_l1.assemble_nat_bc(sd, lambda _: [1, 0], right)

Once the data are assigned, we construct the matrices. The linear system associated with the equation is given as
$$
\left(
\begin{array}{cc}
A & -B^\top \\
B & S
\end{array}
\right)
\left(
\begin{array}{c}
{u} \\
p
\end{array}
\right)
=\left(
\begin{array}{c}
{t}_{\partial} \\
0
\end{array}
\right)
$$
where $A$ is the vector Lagrange stiffness matrix, $B$ represents the (weak) divergence coupling $\int_\Omega q\, \nabla \cdot {u}\, dx$ for continuous $q$, obtained by projecting the broken divergence of ${u}$ onto the piecewise linear space and back onto the continuous pressure space, $S$ is the Brezzi-Pitkäranta stabilization matrix, and ${t}_{\partial}$ is the vector associated with the natural boundary condition. To construct the saddle-point problem, we rely on the `scipy.sparse` function `block_array`.

In [5]:
mu = 1
lambda_ = 0
delta = 1  # stabilization weight

# data for the viscous term
data = pp.initialize_data({}, key, {pg.LAME_LAMBDA: lambda_, pg.LAME_MU: mu})

# viscous term
A = vec_l1.assemble_stiff_matrix_elasticity(sd, data)

# projections needed to build the velocity-pressure coupling term
proj_l1_p1 = l1.proj_to_PwPolynomials(sd)
proj_p0_p1 = p0.proj_to_higher_PwPolynomials(sd)

mass_p1 = p1.assemble_mass_matrix(sd)
div_b = vec_l1.assemble_broken_div_matrix(sd)

B = proj_l1_p1.T @ mass_p1 @ proj_p0_p1 @ div_b

# Brezzi-Pitkäranta pressure stabilization, scaled with the cell diameter
stab = delta * np.square(sd.cell_diameters())
data_stab = pp.initialize_data({}, key, {pg.WEIGHT: stab})

S = l1.assemble_stiff_matrix(sd, data_stab)

# assemble the saddle point problem
spp = sps.block_array([[A, -B.T], [B, S]]).tocsc()

# assemble the right-hand side
rhs = np.zeros(dofs.sum())
rhs[: dofs[0]] = rhs_v

We solve the linear system and extract the two solutions ${u}$ and $p$.

In [6]:
# solve the problem
ls = pg.LinearSystem(spp, rhs)
ls.flag_ess_bc(ess_dofs, ess_val)
up = ls.solve()

# split the solution into the components
u, p = np.split(up, [dofs[0]])

Since ${u}$ has two components per node, we reshape it for visualization purposes. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [7]:
# reshape the velocity as a vector field
u_3d = np.zeros((pg.AMBIENT_DIM, sd.num_nodes))
u_3d[: sd.dim, :] = u.reshape((sd.dim, -1))

# export the solution
folder_name = os.path.join(os.getcwd(), key)
file_name = "sol"

save = pp.Exporter(sd, file_name, folder_name=folder_name)
save.write_vtu(data_pt=[("p", p), ("u", u_3d)])

A representation of the computed solution is given below.

<div style="text-align: center;">
  <img src="../fig/stokes_stab.png" alt="Stokes solution"/>
</div>

In [8]:
# Consistency check
assert np.isclose(np.linalg.norm(u), 1.224955308703262)
assert np.isclose(np.linalg.norm(p), 6.627192586685019)